# Suite BRE — The POSIX BRE dialect

`Expressions.POSIX.BRE` models POSIX's basic regular expressions (IEEE Std 1003.1, XBD 9.3, its grammar 9.5.2), as grep
reads its patterns, a newline-separated list of them, and as C++'s `std::regex` reads them under its `basic` and `grep`
grammars. Its kinds are validated as its grammar has them, bind nothing, cover POSIX.1's four editions, print as a BRE
writes them, are refused by evaluation by name, and are checked complete against XBD 9.5 kept as data.

In [ ]:
import { readFileSync as readGrammar } from "node:fs";
import { Framework } from "@mbse/expressions";
import * as BRE from "../src/POSIX/BRE/index.js";
import * as ERE from "@mbse/expressions/POSIX/ERE";
import { Simplifiers as FE, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const [B, BText, R] = [BRE.Expressions, BRE.Text, ERE.Expressions];

const store: any = new Builders.Store(); // the POSIX family's dialects, ERE's install first: the installs run in any order
for (const install of [Framework.install, ERE.install, BRE.install]) store.transaction(install);
store.transaction(BRE.define); // its editions

/** The term value `spec` writes, the root of `dialect`'s holder, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = B.DIALECT): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)))));
}

const validate = (spec: unknown): string[] => B.DIALECT.validate(term(spec));
const text = (spec: unknown): string => BText.ToText(term(spec));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
/** A character, of its code point, and its spelling. */
const char = (c: string, spelling?: string) => (u: any) => u.character(
  (r: any) => spelling === undefined ? r.value(BigInt(c.codePointAt(0) as number)) : r.raw(spelling).value(BigInt(c.codePointAt(0) as number)));
const alt = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const capture = (...alternatives: unknown[]) => (u: any) => u.capturinggroup((r: any) => r.alternatives(alternatives));

/** A quantifier: `element\{least,most\}`, as `spelling` writes it where given. */
function repeat(element: unknown, least: bigint, most?: bigint, spelling?: string): unknown {
  return (u: any) => u.quantifier((r: any) => {
    r = r.min(least).element(element);
    r = most === undefined ? r : r.max(most);
    return spelling === undefined ? r : r.raw(spelling);
  });
}

const cls = (elements: unknown[], negate = false) => (u: any) => u.characterclass((r: any) => r.negate(negate).elements(elements));
const span = (low: unknown, high: unknown) => (u: any) => u.characterclassrange((r: any) => r.min(low).max(high));
const chars = (kind: string) => (u: any) => u.characterset((r: any) => r.setKind(kind));
const dot = chars("any");
const anchor = (kind: string) => (u: any) => u.assertion((r: any) => r.assertionKind(kind));
const backref = (n: bigint) => (u: any) => u.backreference((r: any) => r.ref(n));
const named = (kind: string, name: string) => (u: any) => u[kind]((r: any) => r.name(name)); // `[:alpha:]`, `[.a.]`, `[=a=]`
const patterns = (...specs: unknown[]) => (u: any) => u.patternlist((r: any) => r.patterns(specs));
const error = (source: string) => (u: any) => u.error((r: any) => r.text(source));
const missing = (kind: string) => (u: any) => u.missing((r: any) => r.kind(kind));
const abc = pattern(alt(char("a"), char("b")), alt(char("c"))); // `ab\|c`

<a id="BRE-01"></a>
## BRE-01 · Its union and holder: its kinds and invalid syntax, a branch of the framework's terms, reading its family's other dialect; a holder refining the framework's, round-tripped through JSON

In [ ]:
{
  assert(same(B.TAGS, ["patternlist", "pattern", "alternative", "capturinggroup", "quantifier", "characterclass",
    "characterclassrange", "characterset", "character", "assertion", "backreference", "exclass", "collatingelement",
    "equivalence"]) && B.HELPERS.length === 0 && B.SINCE.size === 0);
  assert(same([...B.DIALECT.kinds().keys()], union(B.DIALECT.union)) && same(union(B.DIALECT.union), [...B.TAGS, "error", "missing"]));
  assert(B.DIALECT.name() === "POSIX BRE" && B.DIALECT.tag === "posix_bre" && B.DIALECT.union === "Expressions.POSIX.BRE.Term");
  assert(union("Expressions.Framework.Term").includes("posix_bre") && union("Expressions.Framework.Dialect").includes("posix_bre"));
  assert(same(B.DIALECT.reads(), [R.DIALECT]) && same(R.DIALECT.reads(), [B.DIALECT])); // its family's other dialect
  const Holder: any = store.registered("Expressions.POSIX.BRE.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(B.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const holder = store.transaction(() => store.Expressions.POSIX.BRE.Expression((e: any) => e.root(B.DIALECT.spec(abc))));
  assert(Validators.Validate(store)(Holder, holder).length === 0 && B.DIALECT.validate(holder).length === 0);
  assert(F.dialectOf(F.term(holder)) === B.DIALECT.union && F.kindOf(F.term(holder))[0] === "pattern");
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  raises(Error, () => store.transaction(() => // its root holds its own terms: a BRE has no group
    store.Expressions.POSIX.BRE.Expression((e: any) => e.root(B.DIALECT.spec((u: any) => u.group((r: any) => r))))),
  "'group' is not a member of a union value");
  assert(B.DIALECT.validate(term(repeat(repeat(char("a"), 1n), 0n), R.DIALECT)).length === 0); // an ERE's term, as POSIX ERE reads it
}

<a id="BRE-02"></a>
## BRE-02 · Its kinds: their properties in order, each term property its Term; forms and make round-trip

In [ ]:
{
  const props = (name: string): string[] => [...store.registered(`Expressions.POSIX.BRE.${name}`).properties.keys()];
  assert(same(props("PatternList"), ["patterns"]) && same(props("Pattern"), ["alternatives"]) && same(props("CapturingGroup"), ["alternatives"]));
  assert(same(props("Alternative"), ["elements"]) && same(props("Quantifier"), ["raw", "min", "max", "element"])); // never lazy
  assert(same(props("CharacterClass"), ["negate", "elements"]) && same(props("CharacterClassRange"), ["min", "max"]));
  assert(same(props("CharacterSet"), ["setKind"]) && same(props("Character"), ["raw", "value"]) && same(props("Assertion"), ["assertionKind"]));
  assert(same(props("Backreference"), ["ref"]) && same(props("ExClass"), ["name"]) && same(props("CollatingElement"), ["name"])
    && same(props("Equivalence"), ["name"]));
  const prop = (name: string, member: string): any => store.registered(`Expressions.POSIX.BRE.${name}`).properties.get(member).schema;
  const Term = store.registered(B.DIALECT.union);
  assert(prop("Quantifier", "element") === Term && prop("CharacterClassRange", "min") === Term && prop("CharacterClassRange", "max") === Term);
  assert(prop("PatternList", "patterns").element === Term && prop("Pattern", "alternatives").element === Term
    && prop("Alternative", "elements").element === Term);
  const made = F.form(term(pattern(alt(repeat(char("a"), 2n, 3n), backref(1n)))));
  assert(made.kind === "pattern" && same(made.arguments.map((a) => F.kindOf(a)[0]), ["alternative"]));
  const quantified = F.form(term(repeat(cls([span(char("a"), char("z"))]), 1n)));
  assert(equal(quantified.attributes, new Map<string, unknown>([["min", 1n]]))
    && same(quantified.arguments.map((a) => F.kindOf(a)[0]), ["characterclass"]));
  for (const spec of [patterns(abc, pattern(alt(dot, anchor("end")))), capture(alt(backref(1n))),
    cls([named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), char("-")], true)]) {
    assert(F.same(term(B.DIALECT.make(F.form(term(spec)))), term(spec)));
  }
}

<a id="BRE-03"></a>
## BRE-03 · Its checks: what each kind's terms may be, as XBD 9.5.2 has them, where an anchor stands, and its attributes' values

In [ ]:
{
  const cases: [unknown, string[]][] = [
    [patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]],
    [pattern(char("a")), ["a pattern's alternatives cannot hold a character"]],
    [alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]],
    [alt(named("exclass", "alpha")), ["an alternative's elements cannot hold an exclass"]],
    [capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]],
    [repeat(repeat(char("a"), 1n), 0n), ["a quantifier's element cannot hold a quantifier"]], // nondupl_BRE BRE_dupl_symbol
    [repeat(anchor("start"), 0n), ["a quantifier's element cannot hold an assertion"]], // `^*`: its `*` a character
    [cls([alt()]), ["a characterclass's elements cannot hold an alternative"]],
    [cls([dot]), ["a characterclass's elements cannot hold a characterset"]], // `.` within brackets is a character
    [span(dot, named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]],
    [repeat(char("a"), -1n), ["a quantifier's min cannot be negative, got -1"]],
    [repeat(char("a"), 3n, 2n), ["a quantifier's max cannot be less than its min, got 2 and 3"]],
    [(u: any) => u.quantifier((r: any) => r.element(char("a"))), ["a quantifier needs a min"]],
    [(u: any) => u.quantifier((r: any) => r.min(1n)), ["a quantifier needs an element"]],
    [chars("digit"), ["CharacterSet.setKind cannot be 'digit'"]],
    [(u: any) => u.characterset((r: any) => r), ["a characterset needs a setKind"]],
    [(u: any) => u.character((r: any) => r.value(0x110000n)), ["a character's value must be a code point, got 1114112"]],
    [(u: any) => u.character((r: any) => r.raw("a")), ["a character needs a value"]],
    [anchor("word"), ["Assertion.assertionKind cannot be 'word'"]],
    [(u: any) => u.assertion((r: any) => r), ["an assertion needs an assertionKind"]],
    [backref(0n), ["a backreference's ref must be 1 to 9, got 0"]],
    [backref(10n), ["a backreference's ref must be 1 to 9, got 10"]], // BACKREF: a single digit
    [named("exclass", "a:b"), ["an exclass's name cannot hold ':'"]],
    [named("collatingelement", "a.="), ["a collatingelement's name cannot hold '.'"]],
    [named("equivalence", ""), ["an equivalence needs a name"]],
    [alt(char("a"), anchor("start")), ["a start assertion stands only where a BRE begins"]], // L_ANCHOR
    [alt(anchor("end"), char("a")), ["an end assertion stands only where a BRE ends"]], // R_ANCHOR
    [capture(alt(anchor("start"), char("a"))), ["a BRE's subexpression holds no anchor"]], // 9.3.8: the implementation's
  ];
  for (const [spec, problems] of cases) assert(same(validate(spec), problems), `${validate(spec)} != ${problems}`);
  for (const spec of [abc, alt(), pattern(alt(anchor("start"), char("a"), anchor("end")), alt(anchor("start"))),
    cls([span(char("a"), named("collatingelement", "z")), char("-")]), repeat(backref(9n), 0n), capture(alt())]) {
    assert(validate(spec).length === 0);
  }
}

<a id="BRE-04"></a>
## BRE-04 · grep's list of patterns: each a BRE of its own, anchored where it begins and ends, printed a line each

In [ ]:
{
  const listed = patterns(pattern(alt(anchor("start"), char("a"))), pattern(alt(char("b"), anchor("end"))), abc);
  assert(validate(listed).length === 0 && text(listed) === "^a\nb$\nab\\|c"); // grep's pattern_list (XCU grep)
  assert(same(validate(patterns(pattern(alt(char("a"), anchor("start"))))), [
    "argument 0: argument 0: a start assertion stands only where a BRE begins"]));
  assert(same([...F.walk(term(patterns(pattern(alt(char("a"))))))].map((t) => F.kindOf(t)[0]), [
    "patternlist", "pattern", "alternative", "character"]));
}

<a id="BRE-05"></a>
## BRE-05 · What it binds: nothing; a subexpression numbered, a backreference resolving it by its number

In [ ]:
{
  assert(!store.registered(B.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
  const grouped = term(pattern(alt(capture(alt(char("a"))), backref(1n))));
  assert(SY.free(grouped, B.DIALECT).size === 0 && B.DIALECT.validate(grouped).length === 0);
  assert(same([...F.walk(grouped)].map((t) => F.kindOf(t)[0]), ["pattern", "alternative", "capturinggroup", "alternative",
    "character", "backreference"]));
}

<a id="BRE-06"></a>
## BRE-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
{
  assert(same(validate(error("\\(")), ["invalid syntax '\\\\('"]) && text(error("\\(")) === "\\(");
  assert(same(validate(missing("atom")), ["missing atom"]) && text(alt(char("a"), missing("atom"))) === "a");
  assert(same(validate(pattern(alt(char("a"), error("[")))), ["argument 0: argument 1: invalid syntax '['"]));
  assert(text(pattern(alt(char("a"), error("[z")))) === "a[z");
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "POSIX BRE error has no text to print");
}

<a id="BRE-07"></a>
## BRE-07 · Its editions: POSIX.1's four, each by its designation and common names, the ISO/IEC designations C++ cites among them, each with every kind

In [ ]:
{
  const bre_ = store.Instances.Expressions.POSIX.BRE;
  assert(equal([...FS.standards(bre_).values()].map((s) => FS.tags(s)), [
    ["IEEE Std 1003.1-2001", "POSIX.1-2001", "ISO/IEC 9945:2003"],
    ["IEEE Std 1003.1-2008", "POSIX.1-2008", "ISO/IEC/IEEE 9945:2009"],
    ["IEEE Std 1003.1-2017", "POSIX.1-2017"], ["IEEE Std 1003.1-2024", "POSIX.1-2024"]]));
  assert(equal(B.EDITIONS[1], ["IEEE Std 1003.1-2008", 2008n, ["POSIX.1-2008", "ISO/IEC/IEEE 9945:2009"], "POSIX_2008"]));
  assert(FS.find(bre_, "ISO/IEC/IEEE 9945:2009") === FS.find(bre_, "POSIX.1-2008")); // the designation C++23 cites
  const edition = FS.properties(FS.find(bre_, "POSIX.1-2024")!);
  assert(equal(edition.get("bodies"), ["org.ieee"]) && edition.get("number") === "1003.1" && equal(edition.get("revision"), ["year", 2024n]));
  const names = ["PatternList", "Pattern", "Alternative", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "Backreference", "ExClass", "CollatingElement", "Equivalence"];
  assert([...FS.standards(bre_).values()].every((e) => same(FS.kinds(e).map((s: any) => s.name),
    names.map((name) => `Expressions.POSIX.BRE.${name}`))));
  assert(store.transaction(BRE.define)[0].get("POSIX_2008") === FS.find(bre_, "POSIX.1-2008")); // run again, it writes nothing
}

<a id="BRE-08"></a>
## BRE-08 · Printing: every kind as a BRE writes it, its special characters escaped, a spelling as written; an ERE's term as an ERE

In [ ]:
{
  const every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0n), repeat(char("c"), 1n),
    repeat(char("d"), 0n, 1n), repeat(char("e"), 2n, 2n), repeat(char("f"), 2n), repeat(char("g"), 2n, 5n),
    cls([span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"), named("equivalence", "e"), char("]")], true),
    dot, backref(1n), anchor("end")));
  assert(text(every) === "^\\(a\\|\\.\\)b*c\\{1,\\}d\\{0,1\\}e\\{2\\}f\\{2,\\}g\\{2,5\\}[^a-z[:digit:][.ch.][=e=]]].\\1$");
  assert(text(abc) === "ab\\|c"); // POSIX.1-2024's alternation
  assert(text(pattern(alt(char("+"), char("*"), char("?"), char("$"), char("\n"), cls([char("\\"), char("*")])))) === "+\\*?\\$\n[\\*]");
  assert(text(pattern(alt(repeat(char("a"), 1n, undefined, "\\+"), char("x", "\\x")))) === "a\\+\\x"); // as written
  assert(BText.ToText(term(pattern(alt(repeat(char("a"), 0n, 1n), capture(alt(char("b"))))), R.DIALECT)) === "a?(b)"); // an ERE's
}

<a id="BRE-09"></a>
## BRE-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
{
  const simplify = new FE.Simplifier(new FE.Interpreter(B.DIALECT, new Map(), { folded: new Set() }), { literal: () => null }); // no kind folded: each stays, named
  for (const spec of [patterns(abc), abc, char("a"), cls([]), backref(1n)]) {
    raises(NotEvaluated, () => simplify.run(term(spec)).evaluate(), `POSIX BRE ${F.kindOf(term(spec))[0]} is not simplified`);
  }
  raises(NotEvaluated, () => simplify.run(term(error("\\("))).evaluate(), "POSIX BRE error is not simplified");
}

<a id="BRE-10"></a>
## BRE-10 · Completeness: grammars/posix-regex.json maps every production and token of XBD 9.5, POSIX.1-2024's too, and every kind is mapped

In [ ]:
{
  const grammar = globalThis.JSON.parse(readGrammar("../../grammars/posix-regex.json", "utf8"));
  const nodes: any[] = grammar.nodes;
  assert(nodes.length === 57 && new Set(nodes.map((n) => n.node)).size === 57 && grammar.checked.includes("57 productions and tokens"));
  assert(nodes.every((n) => {
    const keys = Object.keys(n).filter((k) => k !== "note").sort().join(",");
    return keys === "kinds,node,source" || keys === "node,not,source";
  }));
  const mapped = new Set<string>(nodes.flatMap((n) => n.kinds ?? []));
  assert(equal([...mapped].filter((kind) => kind.startsWith("BRE.")).sort(), B.TAGS.map((tag) => `BRE.${tag}`).sort())); // every kind, exactly
  const by = new Map(nodes.map((n) => [n.node, n]));
  const renamed = [["RE_expression", "BRE_branch"], ["simple_RE", "simple_BRE"], ["nondupl_RE", "nondupl_BRE"],
    ["one_char_or_coll_elem_RE", "one_char_or_coll_elem_BRE"], ["RE_dupl_symbol", "BRE_dupl_symbol"]];
  assert(renamed.every(([old, renamed_]) => equal(by.get(old).kinds, by.get(renamed_).kinds))); // POSIX.1-2024's names, alike
  assert(["Back_qm", "Back_plus", "Back_bar"].every((token) => "not" in by.get(token))); // its `\?`, `\+`, `\|`: a raw, a printing
  assert(same(by.get("pattern_list").kinds, ["BRE.patternlist", "ERE.patternlist"])); // grep's, XCU grep
}